# ЛР3 — аудиторный demo-notebook

**Тема:** от `fit()` к явному шагу обучения в PyTorch  
**Формат:** интерактивный разбор перед началом student-notebook практики B  
**Данные:** маленькая синтетическая **уже подготовленная** матрица признаков

> **Центральный вопрос:** что становится нашей явной ответственностью, когда вместо готового `.fit()` мы сами организуем training loop?

Этот notebook **не является решением ЛР3**. В нём другие данные, другие размеры сети и нет полного training loop, learning curves, controlled experiment и финального test. После demo мы открываем `student.ipynb` ЛР3 и продолжаем работу уже в основной лабораторной.


> **Как работаем с demo.** Notebook открыт у каждого студента. Не перепечатывайте демонстрационный код: сначала отвечайте на вопросы и прогнозируйте формы/изменения, затем выполняйте ячейки синхронно с разбором.


## 0. Окружение

ЛР1 дала нам привычку следить за формами. ЛР2 — следить за тем, откуда пришла информация. В ЛР3 добавляется ещё один вопрос:

> **что именно изменила эта строка — вычислила выход, посчитала градиенты или изменила параметры?**

GPU для demo не нужен.


In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.exceptions import ConvergenceWarning

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

DEMO_SEED = 7
np.random.seed(DEMO_SEED)
torch.manual_seed(DEMO_SEED)
torch.set_num_threads(1)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)


# 1. Представим, что preprocessing уже выполнен

В student-notebook практики B данные приходят из ЛР2 уже в виде числовых массивов. Здесь сделаем то же самое искусственно: создадим небольшую матрицу `X_demo` без пропусков и категориальных строк.

До запуска ответьте:

- сколько объектов будет в train;
- сколько признаков у одного объекта;
- изменится ли число признаков, когда мы начнём выдавать данные по batch.


In [ ]:
rng = np.random.default_rng(DEMO_SEED)

X_demo = rng.normal(size=(64, 4)).astype(np.float32)
score = (
    1.2 * X_demo[:, 0]
    - 0.9 * X_demo[:, 1]
    + 0.8 * X_demo[:, 2] * X_demo[:, 3]
    + 0.15 * rng.normal(size=len(X_demo))
)
y_demo = (score > np.median(score)).astype(np.int64)

order = rng.permutation(len(X_demo))
train_idx, val_idx = order[:48], order[48:]

X_train = X_demo[train_idx]
y_train = y_demo[train_idx]
X_val = X_demo[val_idx]
y_val = y_demo[val_idx]

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("X_val:  ", X_val.shape)
print("Доля класса 1 в train:", round(float(y_train.mean()), 3))


# 2. Сначала — привычный высокий уровень: `MLPClassifier.fit()`

С точки зрения пользователя sklearn всё обучение можно запустить одной командой. Это удобно — и именно поэтому полезно спросить:

> **что должно происходить внутри `.fit()`, хотя мы этого пока не пишем?**

Не пытаемся разобрать внутренности sklearn построчно. Наша цель — составить список скрытых обязанностей, которые скоро станут явными в PyTorch.


In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=ConvergenceWarning)
    high_level_mlp = MLPClassifier(
        hidden_layer_sizes=(8,),
        activation="relu",
        solver="adam",
        learning_rate_init=1e-2,
        batch_size=8,
        max_iter=600,
        random_state=DEMO_SEED,
    )
    high_level_mlp.fit(X_train, y_train)

val_pred = high_level_mlp.predict(X_val)
print("Validation accuracy:", round(accuracy_score(y_val, val_pred), 3))
print("Вызов обучения для нас выглядел как одна строка: .fit(...)")


### Что `.fit()` скрывает от нас

В собственной реализации нам придётся явно организовать хотя бы такую последовательность:

```text
данные → batches → model → logits → loss
                         ↓
                    gradients
                         ↓
                  optimizer step
```

Пока не пишем цикл. Разберём каждый кирпич отдельно.


# 3. `TensorDataset` и `DataLoader`: из матрицы — в batch

`Tensor` здесь можно воспринимать как числовой массив PyTorch. `TensorDataset` хранит соответствующие друг другу `X` и `y`, а `DataLoader` выдаёт их небольшими пачками.

В demo используем `batch_size = 8` — специально не 32, как в student-notebook практики B.


In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

train_dataset = TensorDataset(X_train_t, y_train_t)
generator = torch.Generator().manual_seed(DEMO_SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    generator=generator,
    num_workers=0,
)

xb, yb = next(iter(train_loader))
print("X batch:", xb.shape, xb.dtype)
print("y batch:", yb.shape, yb.dtype)


### Связь с ЛР1

`xb.shape == (8, 4)` — снова знакомая batch-матрица `(B, D)`.

- `B = 8` — сколько объектов сейчас обрабатываем вместе;
- `D = 4` — сколько входных признаков у каждого объекта.

Если скрытый слой выдаёт 8 новых признаков, концептуально мы снова делаем преобразование вида `X @ W + b`, а затем ReLU.


# 4. `nn.Module`: описываем саму модель

Сеть в demo:

```text
4 входных признака
       ↓
Linear → 8
       ↓
ReLU
       ↓
Linear → 1
       ↓
один logit на объект
```

Последний выход — **не вероятность**, а raw score, или **logit**. Для batch из восьми объектов ожидаем восемь logits.


In [ ]:
class DemoMLP(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=8, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


torch.manual_seed(DEMO_SEED)
model = DemoMLP(dropout=0.0)
print(model)

logits = model(xb)
print("logits shape:", logits.shape)
print("первые logits:", logits[:4].detach().numpy().round(3))


# 5. Loss и smoke test: проверяем один batch до длинного обучения

В student-notebook практики B используется `BCEWithLogitsLoss`. Она ожидает logits и target одной формы.

Мы **не добавляем `Sigmoid` в `forward`**: `BCEWithLogitsLoss` уже объединяет преобразование logits и binary cross-entropy численно устойчивым способом.

Smoke test отвечает на очень дешёвые вопросы:

- совпадают ли формы logits и target;
- получается ли конечный скалярный loss;
- вообще можно ли провести один forward до того, как мы запустим сотни эпох.


In [ ]:
criterion = nn.BCEWithLogitsLoss()
loss = criterion(logits, yb)

print("logits:", logits.shape)
print("target:", yb.shape)
print("loss shape:", loss.shape)
print("loss:", round(float(loss.item()), 4))

assert logits.shape == yb.shape
assert loss.ndim == 0
assert torch.isfinite(loss)
print("Smoke test пройден.")


# 6. Один шаг обучения: пять строк с разными обязанностями

Теперь главный фрагмент пары:

```python
optimizer.zero_grad()
logits = model(x_batch)
loss = criterion(logits, y_batch)
loss.backward()
optimizer.step()
```

Перед запуском зафиксируйте прогноз:

1. После какой строки появятся градиенты?
2. После какой строки **изменятся параметры модели**?
3. Почему это не одна и та же строка?


In [ ]:
torch.manual_seed(DEMO_SEED)
step_model = DemoMLP(dropout=0.0)
step_optimizer = torch.optim.Adam(step_model.parameters(), lr=1e-2)
step_criterion = nn.BCEWithLogitsLoss()

first_param = next(step_model.parameters())
weights_before = first_param.detach().clone()

step_optimizer.zero_grad()
print("До backward gradient:", first_param.grad)

step_logits = step_model(xb)
step_loss = step_criterion(step_logits, yb)
step_loss.backward()

print("После backward gradient mean |g|:", round(float(first_param.grad.abs().mean()), 6))
weights_after_backward = first_param.detach().clone()
print("Изменились ли веса после backward:", not torch.equal(weights_before, weights_after_backward))

step_optimizer.step()
weights_after_step = first_param.detach().clone()
print("Максимальное изменение веса после step:", round(float((weights_after_step - weights_before).abs().max()), 6))


### Что важно увидеть

- `backward()` **заполняет градиенты** параметров;
- `optimizer.step()` **использует эти градиенты и меняет параметры**;
- поэтому `backward()` и `step()` нельзя считать двумя названиями одного действия.

Остаётся вопрос: зачем каждый batch начинается с `zero_grad()`?


## 6.1. Почему нужен `zero_grad()`: градиенты накапливаются

PyTorch по умолчанию складывает новый gradient с уже лежащим в `.grad`. Это полезная возможность, но в обычном training loop нам обычно нужен gradient именно текущего batch.

Ниже специально **не** вызываем `zero_grad()` между двумя `backward()` на одном и том же batch.


In [ ]:
torch.manual_seed(DEMO_SEED)
acc_model = DemoMLP(dropout=0.0)
acc_criterion = nn.BCEWithLogitsLoss()
acc_param = next(acc_model.parameters())

acc_model.zero_grad()
loss1 = acc_criterion(acc_model(xb), yb)
loss1.backward()
grad_norm_1 = float(acc_param.grad.norm())

loss2 = acc_criterion(acc_model(xb), yb)
loss2.backward()  # намеренно без zero_grad()
grad_norm_2 = float(acc_param.grad.norm())

print("Норма gradient после первого backward:", round(grad_norm_1, 6))
print("Норма gradient после второго backward без zero_grad:", round(grad_norm_2, 6))
print("Отношение:", round(grad_norm_2 / grad_norm_1, 3))


# 7. `train()` и `eval()`: режим модели — часть корректности

В student-notebook практики B controlled experiment использует Dropout. Поэтому важно увидеть режимы **до** полного эксперимента.

С одной и той же моделью и одним и тем же входом:

- в `train()` Dropout активен;
- в `eval()` Dropout отключён.

Отдельно: `torch.no_grad()` не переключает Dropout. Он говорит PyTorch не строить граф для вычисления градиентов. Поэтому при validation обычно нужны **и** `model.eval()`, **и** `torch.no_grad()`.


In [ ]:
torch.manual_seed(DEMO_SEED)
drop_model = DemoMLP(dropout=0.5)
probe = xb[:4]

drop_model.train()
train_out_1 = drop_model(probe).detach()
train_out_2 = drop_model(probe).detach()

drop_model.eval()
with torch.no_grad():
    eval_out_1 = drop_model(probe)
    eval_out_2 = drop_model(probe)

print("max |train pass 1 - train pass 2|:", round(float((train_out_1 - train_out_2).abs().max()), 6))
print("max |eval pass 1 - eval pass 2|: ", round(float((eval_out_1 - eval_out_2).abs().max()), 6))


# 8. Мини-checkpoint

Сначала ответьте **без запуска кода**.

### A. Формы
Дан batch `X` формы `(5, 4)`. Скрытый слой имеет ширину `H=6`, последний слой выдаёт один logit.

- какой формы будет выход скрытого слоя;
- какой формы должны быть финальные logits после `squeeze(1)`;
- какой формы нужен target для `BCEWithLogitsLoss`?

### B. Порядок шага обучения
Расставьте в правильном порядке:

- `loss.backward()`
- `optimizer.step()`
- `model(xb)`
- `optimizer.zero_grad()`
- `criterion(logits, yb)`

### C. Два разных действия
Почему `loss.backward()` не обучает модель до конца сам по себе?

### D. Режимы
Почему validation для модели с Dropout нельзя выполнять, оставив `model.train()`?

### E. Live mini-task
Измените в `DemoMLP` скрытую ширину `8 → 6`. **До запуска** назовите ожидаемую форму logits для batch `(5, 4)`.


# 9. Переход в student-notebook практики B

Теперь открываем **ЛР3 `MLP: sklearn vs PyTorch`**. В отличие от ЛР1–ЛР2, это уже не отдельная домашняя тетрадь после demo: `student.ipynb` является основной экспериментальной работой и начинается на занятии.

Первые действия в основной работе:

1. загрузить артефакт ЛР2 и проверить shapes/dtypes;
2. **не делать новый split**;
3. восстановить Logistic Regression baseline;
4. обучить `MLPClassifier` на тех же prepared arrays;
5. заполнить гипотезы **до** основной PyTorch-модели и controlled experiment.

На demo мы разобрали механику одного шага. В `student.ipynb` вы самостоятельно соберёте `evaluate_model`, полный `train_model`, learning curves, controlled experiment `dropout=0.0 → 0.3`, анализ ошибок и честную финальную оценку на test.

Главный вопрос ЛР3 остаётся тем же:

> **что меняется, когда мы переходим от готовой модели к собственному training loop?**
